# Transcriptional fidelity of patient-derived models in colorectal signet-ring cell carcinoma

### Independent re-analysis of public RNA-sequencing data from GSE279979

**Question:** Which transcriptional programmes distinguish colorectal signet-ring cell carcinoma from adjacent normal tissue, and to what extent are these tumour-associated programmes preserved in patient-derived xenograft models?

Because the public RNA-seq cohort contains only two samples per group, this analysis is exploratory and focuses on transcriptional patterns and pathway-level concordance rather than definitive biomarker discovery.

In [ ]:
counts_raw = pd.read_csv(
    "SRCC_raw_counts.txt.gz",
    sep="\t"
)

print("Shape:", counts_raw.shape)

display(counts_raw.head())

print("\nColumns:")
print(counts_raw.columns.tolist())

## 1. Count matrix and sample structure

The processed featureCounts matrix contains two adjacent-normal samples, two primary SRCC tumour samples, and two patient-derived xenograft (PDX) samples.

The analysis asks two questions:

1. Which transcriptional programmes distinguish primary SRCC from adjacent normal tissue?
2. To what extent are those tumour-associated programmes preserved in PDX models?

In [ ]:
sample_cols = ["N_1", "N_2", "T_1", "T_2", "UT_1", "UT_2"]

counts = counts_raw[
    ["Geneid"] + sample_cols
].copy()

# Remove Ensembl version suffix, e.g. ENSG000001234.5 -> ENSG000001234
counts["Geneid"] = counts["Geneid"].str.split(".").str[0]

counts = counts.set_index("Geneid")

print("Count matrix:", counts.shape)
display(counts.head())

In [ ]:
sample_metadata = pd.DataFrame(
    {
        "condition": [
            "Normal",
            "Normal",
            "Tumor",
            "Tumor",
            "PDX",
            "PDX"
        ],
        "pair": [
            "1",
            "2",
            "1",
            "2",
            "1",
            "2"
        ]
    },
    index=sample_cols
)

display(sample_metadata)

In [ ]:
# Filter genes with almost no information
keep = (counts >= 10).sum(axis=1) >= 2
counts_filt = counts.loc[keep]

print("Genes retained:", counts_filt.shape[0])

# Library-size normalize and log transform for visualization only
libsize = counts_filt.sum(axis=0)

norm = counts_filt.div(libsize, axis=1) * 1e6
log_cpm = np.log2(norm + 1)

print(log_cpm.shape)

In [ ]:
from sklearn.decomposition import PCA

X = log_cpm.T

pca = PCA(n_components=2)
pcs = pca.fit_transform(X)

pca_df = pd.DataFrame(
    pcs,
    columns=["PC1", "PC2"],
    index=X.index
)

pca_df["condition"] = sample_metadata.loc[pca_df.index, "condition"]

display(pca_df)

In [ ]:
plt.figure(figsize=(6,5))

for sample in pca_df.index:
    plt.scatter(
        pca_df.loc[sample, "PC1"],
        pca_df.loc[sample, "PC2"],
        s=100
    )

    plt.text(
        pca_df.loc[sample, "PC1"] + 0.2,
        pca_df.loc[sample, "PC2"] + 0.2,
        sample
    )

plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)"
)

plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)"
)

plt.title("Global transcriptional similarity: Normal, Tumour and PDX")
plt.tight_layout()
plt.show()

## 2. Global transcriptional fidelity of the PDX models

PCA suggested that the PDX samples retain the dominant tumour-associated transcriptional state. To quantify this, transcriptome-wide sample correlations and distances to tumour versus normal expression centroids were calculated.

In [ ]:
corr = log_cpm.corr(method="spearman")

display(corr.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(7,6))

im = ax.imshow(
    corr.values,
    vmin=0,
    vmax=1
)

ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")

ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(
            j,
            i,
            f"{corr.iloc[i,j]:.2f}",
            ha="center",
            va="center"
        )

plt.colorbar(im, ax=ax, label="Spearman correlation")
plt.title("Transcriptome-wide similarity between SRCC samples")
plt.tight_layout()
plt.show()

In [ ]:
normal_centroid = log_cpm[["N_1", "N_2"]].mean(axis=1)
tumor_centroid = log_cpm[["T_1", "T_2"]].mean(axis=1)

from scipy.stats import spearmanr

for sample in ["UT_1", "UT_2"]:
    r_tumor, _ = spearmanr(log_cpm[sample], tumor_centroid)
    r_normal, _ = spearmanr(log_cpm[sample], normal_centroid)

    print(sample)
    print("  similarity to tumour:", round(r_tumor, 3))
    print("  similarity to normal:", round(r_normal, 3))
    print()

## 3. Preservation of tumour-associated transcriptional changes

Global correlation can remain high even between biologically different tissues because most genes are shared.

To test disease-model fidelity more directly, expression changes in primary SRCC relative to adjacent normal tissue were compared with expression changes in PDX relative to adjacent normal tissue.

A faithful model should reproduce both the direction and magnitude of tumour-associated transcriptional changes.

In [ ]:
normal_mean = log_cpm[["N_1", "N_2"]].mean(axis=1)
tumor_mean = log_cpm[["T_1", "T_2"]].mean(axis=1)
pdx_mean = log_cpm[["UT_1", "UT_2"]].mean(axis=1)

effects = pd.DataFrame({
    "Tumor_vs_Normal": tumor_mean - normal_mean,
    "PDX_vs_Normal": pdx_mean - normal_mean
})

display(effects.head())

In [ ]:
from scipy.stats import spearmanr

r, p = spearmanr(
    effects["Tumor_vs_Normal"],
    effects["PDX_vs_Normal"]
)

print("Tumour-change vs PDX-change correlation:", round(r, 3))
print("p-value:", p)

In [ ]:
plt.figure(figsize=(7,6))

plt.scatter(
    effects["Tumor_vs_Normal"],
    effects["PDX_vs_Normal"],
    s=8,
    alpha=0.25
)

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)

plt.xlabel("Primary tumour vs normal\nmean log2(CPM+1) difference")
plt.ylabel("PDX vs normal\nmean log2(CPM+1) difference")

plt.title(
    f"Preservation of tumour-associated transcriptional changes\n"
    f"Spearman r = {r:.3f}"
)

plt.tight_layout()
plt.show()

In [ ]:
strong = effects[
    effects["Tumor_vs_Normal"].abs() > 1
].copy()

same_direction = (
    np.sign(strong["Tumor_vs_Normal"]) ==
    np.sign(strong["PDX_vs_Normal"])
)

print("Genes with |tumour-normal effect| > 1:", len(strong))

print(
    "Same direction in PDX:",
    round(same_direction.mean() * 100, 1),
    "%"
)

In [ ]:
strong = effects[
    effects["Tumor_vs_Normal"].abs() > 1
].copy()

same_direction = (
    np.sign(strong["Tumor_vs_Normal"]) ==
    np.sign(strong["PDX_vs_Normal"])
)

print("Strongly altered genes:", len(strong))
print(
    "Same direction in PDX:",
    round(same_direction.mean() * 100, 1),
    "%"
)

In [ ]:
import mygene

mg = mygene.MyGeneInfo()

ensembl_ids = effects.index.tolist()

mapping = mg.querymany(
    ensembl_ids,
    scopes="ensembl.gene",
    fields="symbol",
    species="human",
    as_dataframe=True
)

mapping = mapping[["symbol"]].dropna()
mapping = mapping[~mapping.index.duplicated(keep="first")]

print("Mapped genes:", len(mapping))
display(mapping.head())

In [ ]:
effects_symbols = effects.join(mapping, how="inner")

effects_symbols = (
    effects_symbols
    .dropna(subset=["symbol"])
    .drop_duplicates(subset=["symbol"])
    .set_index("symbol")
)

print(effects_symbols.shape)
display(effects_symbols.head())

In [ ]:
tumor_rank = (
    effects_symbols["Tumor_vs_Normal"]
    .sort_values(ascending=False)
)

pdx_rank = (
    effects_symbols["PDX_vs_Normal"]
    .sort_values(ascending=False)
)

In [ ]:
import gseapy as gp

tumor_gsea = gp.prerank(
    rnk=tumor_rank,
    gene_sets="MSigDB_Hallmark_2020",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

pdx_gsea = gp.prerank(
    rnk=pdx_rank,
    gene_sets="MSigDB_Hallmark_2020",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

tumor_pathways = tumor_gsea.res2d.copy()
pdx_pathways = pdx_gsea.res2d.copy()

display(
    tumor_pathways[
        ["Term", "NES", "FDR q-val"]
    ].sort_values("FDR q-val").head(15)
)

display(
    pdx_pathways[
        ["Term", "NES", "FDR q-val"]
    ].sort_values("FDR q-val").head(15)
)

In [ ]:
strong = effects[
    effects["Tumor_vs_Normal"].abs() > 1
].copy()

same_direction = (
    np.sign(strong["Tumor_vs_Normal"]) ==
    np.sign(strong["PDX_vs_Normal"])
)

print("Strongly altered genes:", len(strong))
print(
    "Same direction in PDX:",
    round(same_direction.mean() * 100, 1),
    "%"
)

In [ ]:
!pip install -q mygene

In [ ]:
import mygene

mg = mygene.MyGeneInfo()

mapping = mg.querymany(
    effects.index.tolist(),
    scopes="ensembl.gene",
    fields="symbol",
    species="human",
    as_dataframe=True
)

mapping = mapping[["symbol"]].dropna()
mapping = mapping[~mapping.index.duplicated(keep="first")]

effects_symbols = effects.join(mapping, how="inner")

effects_symbols = (
    effects_symbols
    .dropna(subset=["symbol"])
    .sort_values(
        "Tumor_vs_Normal",
        key=lambda x: x.abs(),
        ascending=False
    )
    .drop_duplicates(subset=["symbol"])
    .set_index("symbol")
)

print("Mapped genes:", len(effects_symbols))

In [ ]:
import gseapy as gp

tumor_rank = (
    effects_symbols["Tumor_vs_Normal"]
    .sort_values(ascending=False)
)

pdx_rank = (
    effects_symbols["PDX_vs_Normal"]
    .sort_values(ascending=False)
)

tumor_gsea = gp.prerank(
    rnk=tumor_rank,
    gene_sets="MSigDB_Hallmark_2020",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

pdx_gsea = gp.prerank(
    rnk=pdx_rank,
    gene_sets="MSigDB_Hallmark_2020",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

tumor_pathways = tumor_gsea.res2d.copy()
pdx_pathways = pdx_gsea.res2d.copy()

In [ ]:
display(
    tumor_pathways[
        ["Term", "NES", "FDR q-val"]
    ].sort_values("FDR q-val").head(15)
)

display(
    pdx_pathways[
        ["Term", "NES", "FDR q-val"]
    ].sort_values("FDR q-val").head(15)
)

In [ ]:
pathway_compare = (
    tumor_pathways[
        ["Term", "NES", "FDR q-val"]
    ]
    .merge(
        pdx_pathways[
            ["Term", "NES", "FDR q-val"]
        ],
        on="Term",
        suffixes=("_tumor", "_pdx")
    )
)

display(pathway_compare.head())

In [ ]:
from scipy.stats import spearmanr

pathway_r, pathway_p = spearmanr(
    pathway_compare["NES_tumor"],
    pathway_compare["NES_pdx"]
)

print("Pathway NES concordance:", round(pathway_r, 3))
print("p-value:", pathway_p)

In [ ]:
plt.figure(figsize=(7,6))

plt.scatter(
    pathway_compare["NES_tumor"],
    pathway_compare["NES_pdx"],
    s=45,
    alpha=0.7
)

for _, row in pathway_compare.iterrows():
    if (
        row["FDR q-val_tumor"] < 0.05
        or row["FDR q-val_pdx"] < 0.05
    ):
        plt.text(
            row["NES_tumor"] + 0.03,
            row["NES_pdx"] + 0.03,
            row["Term"],
            fontsize=8
        )

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)

plt.xlabel("Primary tumour vs normal: pathway NES")
plt.ylabel("PDX vs normal: pathway NES")

plt.title(
    f"Preservation of tumour-associated biological programs\n"
    f"Spearman r = {pathway_r:.3f}"
)

plt.tight_layout()
plt.show()

In [ ]:
preserved = pathway_compare[
    (pathway_compare["FDR q-val_tumor"] < 0.05) &
    (pathway_compare["FDR q-val_pdx"] < 0.05) &
    (
        np.sign(pathway_compare["NES_tumor"]) ==
        np.sign(pathway_compare["NES_pdx"])
    )
].copy()

lost = pathway_compare[
    (pathway_compare["FDR q-val_tumor"] < 0.05) &
    (pathway_compare["FDR q-val_pdx"] >= 0.05)
].copy()

pdx_acquired = pathway_compare[
    (pathway_compare["FDR q-val_tumor"] >= 0.05) &
    (pathway_compare["FDR q-val_pdx"] < 0.05)
].copy()

print("Preserved:", len(preserved))
print("Lost:", len(lost))
print("PDX-acquired:", len(pdx_acquired))

print("\nPRESERVED")
display(
    preserved[
        ["Term", "NES_tumor", "NES_pdx"]
    ].sort_values("NES_tumor", ascending=False)
)

print("\nLOST")
display(
    lost[
        ["Term", "NES_tumor", "NES_pdx"]
    ]
)

print("\nPDX-ACQUIRED")
display(
    pdx_acquired[
        ["Term", "NES_tumor", "NES_pdx"]
    ]
)

In [ ]:
print("Pathway NES correlation:", round(pathway_r, 3))
print("Preserved:", len(preserved))
print("Not significant in PDX:", len(lost))
print("PDX-acquired:", len(pdx_acquired))

In [ ]:
libraries = gp.get_library_name(organism="Human")

print("Reactome libraries:")
print([x for x in libraries if "Reactome" in x])

print("\nGO Biological Process libraries:")
print([x for x in libraries if "GO_Biological_Process" in x])

In [ ]:
tumor_reactome = gp.prerank(
    rnk=tumor_rank,
    gene_sets="Reactome_Pathways_2024",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

pdx_reactome = gp.prerank(
    rnk=pdx_rank,
    gene_sets="Reactome_Pathways_2024",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

tumor_reactome_df = tumor_reactome.res2d.copy()
pdx_reactome_df = pdx_reactome.res2d.copy()

In [ ]:
autophagy_terms_tumor = tumor_reactome_df[
    tumor_reactome_df["Term"]
    .str.contains("autoph|lysosom", case=False, regex=True)
][["Term", "NES", "FDR q-val"]]

autophagy_terms_pdx = pdx_reactome_df[
    pdx_reactome_df["Term"]
    .str.contains("autoph|lysosom", case=False, regex=True)
][["Term", "NES", "FDR q-val"]]

print("TUMOUR")
display(
    autophagy_terms_tumor.sort_values(
        "FDR q-val"
    )
)

print("\nPDX")
display(
    autophagy_terms_pdx.sort_values(
        "FDR q-val"
    )
)

In [ ]:
autophagy_compare = (
    autophagy_terms_tumor
    .merge(
        autophagy_terms_pdx,
        on="Term",
        suffixes=("_tumor", "_pdx")
    )
)

display(autophagy_compare)

In [ ]:
tumor_go = gp.prerank(
    rnk=tumor_rank,
    gene_sets="GO_Biological_Process_2025",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

pdx_go = gp.prerank(
    rnk=pdx_rank,
    gene_sets="GO_Biological_Process_2025",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

tumor_go_df = tumor_go.res2d.copy()
pdx_go_df = pdx_go.res2d.copy()

In [ ]:
tumor_go_autophagy = tumor_go_df[
    tumor_go_df["Term"]
    .str.contains(
        "autoph|lysosom|phagophore",
        case=False,
        regex=True
    )
][["Term", "NES", "FDR q-val"]]

pdx_go_autophagy = pdx_go_df[
    pdx_go_df["Term"]
    .str.contains(
        "autoph|lysosom|phagophore",
        case=False,
        regex=True
    )
][["Term", "NES", "FDR q-val"]]

print("TUMOUR GO")
display(
    tumor_go_autophagy
    .sort_values("FDR q-val")
    .head(20)
)

print("\nPDX GO")
display(
    pdx_go_autophagy
    .sort_values("FDR q-val")
    .head(20)
)

In [ ]:
tumor_go = gp.prerank(
    rnk=tumor_rank,
    gene_sets="GO_Biological_Process_2025",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

pdx_go = gp.prerank(
    rnk=pdx_rank,
    gene_sets="GO_Biological_Process_2025",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

tumor_go_df = tumor_go.res2d.copy()
pdx_go_df = pdx_go.res2d.copy()

print("Tumour GO pathways:", tumor_go_df.shape)
print("PDX GO pathways:", pdx_go_df.shape)

In [ ]:
!pip install -q pandas==2.2.3 gseapy mygene

In [ ]:
counts_raw = pd.read_csv(
    "SRCC_raw_counts.txt.gz",
    sep="\t"
)

sample_cols = ["N_1", "N_2", "T_1", "T_2", "UT_1", "UT_2"]

counts = counts_raw[
    ["Geneid"] + sample_cols
].copy()

counts["Geneid"] = counts["Geneid"].str.split(".").str[0]

counts = counts.set_index("Geneid")

# remove very low-information genes
keep = (counts >= 10).sum(axis=1) >= 2
counts_filt = counts.loc[keep]

# normalize for descriptive/pathway analysis
libsize = counts_filt.sum(axis=0)
norm = counts_filt.div(libsize, axis=1) * 1e6
log_cpm = np.log2(norm + 1)

print("Genes retained:", log_cpm.shape[0])

In [ ]:
normal_mean = log_cpm[["N_1", "N_2"]].mean(axis=1)
tumor_mean = log_cpm[["T_1", "T_2"]].mean(axis=1)
pdx_mean = log_cpm[["UT_1", "UT_2"]].mean(axis=1)

effects = pd.DataFrame({
    "Tumor_vs_Normal": tumor_mean - normal_mean,
    "PDX_vs_Normal": pdx_mean - normal_mean
})

print(effects.shape)

In [ ]:
tumor_rank = (
    effects_symbols["Tumor_vs_Normal"]
    .sort_values(ascending=False)
)

pdx_rank = (
    effects_symbols["PDX_vs_Normal"]
    .sort_values(ascending=False)
)

print("tumor_rank:", len(tumor_rank))
print("pdx_rank:", len(pdx_rank))

In [ ]:
import gseapy as gp

tumor_reactome = gp.prerank(
    rnk=tumor_rank,
    gene_sets="Reactome_Pathways_2024",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

pdx_reactome = gp.prerank(
    rnk=pdx_rank,
    gene_sets="Reactome_Pathways_2024",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

tumor_reactome_df = tumor_reactome.res2d.copy()
pdx_reactome_df = pdx_reactome.res2d.copy()

print("Reactome done")

In [ ]:
tumor_autophagy = tumor_reactome_df[
    tumor_reactome_df["Term"].str.contains(
        "autoph|lysosom",
        case=False,
        regex=True
    )
][["Term", "NES", "FDR q-val"]]

pdx_autophagy = pdx_reactome_df[
    pdx_reactome_df["Term"].str.contains(
        "autoph|lysosom",
        case=False,
        regex=True
    )
][["Term", "NES", "FDR q-val"]]

print("PRIMARY TUMOUR")
display(tumor_autophagy.sort_values("FDR q-val"))

print("\nPDX")
display(pdx_autophagy.sort_values("FDR q-val"))

In [ ]:
autophagy_compare = tumor_autophagy.merge(
    pdx_autophagy,
    on="Term",
    suffixes=("_tumor", "_pdx")
)

display(
    autophagy_compare.sort_values("FDR q-val_tumor")
)

In [ ]:
tumor_go = gp.prerank(
    rnk=tumor_rank,
    gene_sets="GO_Biological_Process_2025",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

pdx_go = gp.prerank(
    rnk=pdx_rank,
    gene_sets="GO_Biological_Process_2025",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

tumor_go_df = tumor_go.res2d.copy()
pdx_go_df = pdx_go.res2d.copy()

In [ ]:
tumor_go_autophagy = tumor_go_df[
    tumor_go_df["Term"].str.contains(
        "autoph|lysosom|phagophore",
        case=False,
        regex=True
    )
][["Term", "NES", "FDR q-val"]]

pdx_go_autophagy = pdx_go_df[
    pdx_go_df["Term"].str.contains(
        "autoph|lysosom|phagophore",
        case=False,
        regex=True
    )
][["Term", "NES", "FDR q-val"]]

print("TUMOUR GO")
display(tumor_go_autophagy.sort_values("FDR q-val").head(20))

print("\nPDX GO")
display(pdx_go_autophagy.sort_values("FDR q-val").head(20))

In [ ]:
print("Gene-effect correlation:", round(r, 3))
print("Strong genes:", len(strong))
print("Same direction:", round(same_direction.mean()*100, 1), "%")

print("Pathway NES correlation:", round(pathway_r, 3))

print("Preserved significant pathways:", len(preserved))
print("Tumour-significant, PDX non-significant:", len(lost))
print("PDX-acquired significant pathways:", len(pdx_acquired))

In [ ]:
from scipy.stats import spearmanr
import numpy as np

# 1. Gene-level fidelity
r, gene_p = spearmanr(
    effects["Tumor_vs_Normal"],
    effects["PDX_vs_Normal"]
)

strong = effects[
    effects["Tumor_vs_Normal"].abs() > 1
].copy()

same_direction = (
    np.sign(strong["Tumor_vs_Normal"]) ==
    np.sign(strong["PDX_vs_Normal"])
)

# 2. Rebuild pathway comparison
pathway_compare = (
    tumor_pathways[
        ["Term", "NES", "FDR q-val"]
    ]
    .merge(
        pdx_pathways[
            ["Term", "NES", "FDR q-val"]
        ],
        on="Term",
        suffixes=("_tumor", "_pdx")
    )
)

# 3. Pathway-level fidelity
pathway_r, pathway_p = spearmanr(
    pathway_compare["NES_tumor"],
    pathway_compare["NES_pdx"]
)

# 4. Classify pathways
preserved = pathway_compare[
    (pathway_compare["FDR q-val_tumor"] < 0.05) &
    (pathway_compare["FDR q-val_pdx"] < 0.05) &
    (
        np.sign(pathway_compare["NES_tumor"]) ==
        np.sign(pathway_compare["NES_pdx"])
    )
].copy()

lost = pathway_compare[
    (pathway_compare["FDR q-val_tumor"] < 0.05) &
    (pathway_compare["FDR q-val_pdx"] >= 0.05)
].copy()

pdx_acquired = pathway_compare[
    (pathway_compare["FDR q-val_tumor"] >= 0.05) &
    (pathway_compare["FDR q-val_pdx"] < 0.05)
].copy()

# 5. Final summary
print("Gene-effect correlation:", round(r, 3))
print("Strongly altered genes:", len(strong))
print("Same direction in PDX:", round(same_direction.mean()*100, 1), "%")
print()
print("Pathway NES correlation:", round(pathway_r, 3))
print("Preserved significant pathways:", len(preserved))
print("Tumour-significant / PDX non-significant:", len(lost))
print("PDX-acquired significant pathways:", len(pdx_acquired))

In [ ]:
import gseapy as gp

# Recreate ranked lists if needed
if "tumor_rank" not in globals() or "pdx_rank" not in globals():
    tumor_rank = (
        effects_symbols["Tumor_vs_Normal"]
        .sort_values(ascending=False)
    )

    pdx_rank = (
        effects_symbols["PDX_vs_Normal"]
        .sort_values(ascending=False)
    )

# Hallmark GSEA
tumor_gsea = gp.prerank(
    rnk=tumor_rank,
    gene_sets="MSigDB_Hallmark_2020",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

pdx_gsea = gp.prerank(
    rnk=pdx_rank,
    gene_sets="MSigDB_Hallmark_2020",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

tumor_pathways = tumor_gsea.res2d.copy()
pdx_pathways = pdx_gsea.res2d.copy()

print("Tumour pathways:", len(tumor_pathways))
print("PDX pathways:", len(pdx_pathways))

In [ ]:
from scipy.stats import spearmanr
import numpy as np

# Gene-level fidelity
r, gene_p = spearmanr(
    effects["Tumor_vs_Normal"],
    effects["PDX_vs_Normal"]
)

strong = effects[
    effects["Tumor_vs_Normal"].abs() > 1
].copy()

same_direction = (
    np.sign(strong["Tumor_vs_Normal"]) ==
    np.sign(strong["PDX_vs_Normal"])
)

# Pathway comparison
pathway_compare = (
    tumor_pathways[
        ["Term", "NES", "FDR q-val"]
    ]
    .merge(
        pdx_pathways[
            ["Term", "NES", "FDR q-val"]
        ],
        on="Term",
        suffixes=("_tumor", "_pdx")
    )
)

pathway_r, pathway_p = spearmanr(
    pathway_compare["NES_tumor"],
    pathway_compare["NES_pdx"]
)

preserved = pathway_compare[
    (pathway_compare["FDR q-val_tumor"] < 0.05) &
    (pathway_compare["FDR q-val_pdx"] < 0.05) &
    (
        np.sign(pathway_compare["NES_tumor"]) ==
        np.sign(pathway_compare["NES_pdx"])
    )
].copy()

lost = pathway_compare[
    (pathway_compare["FDR q-val_tumor"] < 0.05) &
    (pathway_compare["FDR q-val_pdx"] >= 0.05)
].copy()

pdx_acquired = pathway_compare[
    (pathway_compare["FDR q-val_tumor"] >= 0.05) &
    (pathway_compare["FDR q-val_pdx"] < 0.05)
].copy()

print("Gene-effect correlation:", round(r, 3))
print("Strongly altered genes:", len(strong))
print("Same direction in PDX:", round(same_direction.mean()*100, 1), "%")
print()
print("Pathway NES correlation:", round(pathway_r, 3))
print("Preserved significant pathways:", len(preserved))
print("Tumour-significant / PDX non-significant:", len(lost))
print("PDX-acquired significant pathways:", len(pdx_acquired))

In [ ]:
# Direct PDX vs primary tumour difference
direct_effect = pdx_mean - tumor_mean

print("Median absolute PDX-vs-tumour difference:",
      round(direct_effect.abs().median(), 3))

print("90th percentile absolute difference:",
      round(direct_effect.abs().quantile(0.90), 3))

print("Genes with |PDX - tumour| > 1:",
      (direct_effect.abs() > 1).sum())

In [ ]:
direct_rank = (
    effects_symbols["PDX_vs_Normal"]
    - effects_symbols["Tumor_vs_Normal"]
).sort_values(ascending=False)

direct_gsea = gp.prerank(
    rnk=direct_rank,
    gene_sets="MSigDB_Hallmark_2020",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

direct_pathways = (
    direct_gsea.res2d
    .sort_values("FDR q-val")
    .reset_index(drop=True)
)

display(
    direct_pathways[
        ["Term", "NES", "FDR q-val"]
    ].head(15)
)

print(
    "Significant PDX-vs-tumour pathways:",
    (direct_pathways["FDR q-val"] < 0.05).sum()
)

In [ ]:
direct_effect = pdx_mean - tumor_mean

print(
    "Median absolute PDX-vs-tumour difference:",
    round(direct_effect.abs().median(), 3)
)

print(
    "Genes with |PDX - tumour| > 1:",
    (direct_effect.abs() > 1).sum()
)

In [ ]:
# Transcriptional Fidelity of Patient-Derived Xenograft Models in Colorectal Signet-Ring Cell Carcinoma

Independent exploratory re-analysis of public bulk RNA-seq data from GSE279979.

## Research question

How faithfully do colorectal signet-ring cell carcinoma (SRCC) patient-derived xenograft models preserve the transcriptional state and biological programmes of the primary human tumour?

The public dataset contains only two adjacent-normal, two primary-tumour and two PDX samples. This analysis is therefore descriptive and hypothesis-generating rather than a definitive validation study.

## 1. Dataset and preprocessing

## 2. Global transcriptional similarity

## 3. Preservation of tumour-associated gene-expression changes

## 4. Preservation of tumour-associated biological programmes

## 5. Direct PDX versus primary-tumour comparison

## 6. Autophagy-related programme analysis

## 7. Interpretation and limitations

In [ ]:
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

r, p = spearmanr(
    effects["Tumor_vs_Normal"],
    effects["PDX_vs_Normal"]
)

plt.figure(figsize=(7,6))

plt.scatter(
    effects["Tumor_vs_Normal"],
    effects["PDX_vs_Normal"],
    s=8,
    alpha=0.25
)

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)

plt.xlabel("Primary tumour vs normal\nmean log2(CPM+1) difference")
plt.ylabel("PDX vs normal\nmean log2(CPM+1) difference")

plt.title(
    "Preservation of tumour-associated transcriptional changes\n"
    f"Spearman r = {r:.3f}"
)

plt.tight_layout()

plt.savefig(
    "SRCC_gene_effect_fidelity.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from scipy.stats import spearmanr

pathway_r, pathway_p = spearmanr(
    pathway_compare["NES_tumor"],
    pathway_compare["NES_pdx"]
)

plt.figure(figsize=(7,6))

plt.scatter(
    pathway_compare["NES_tumor"],
    pathway_compare["NES_pdx"],
    s=45,
    alpha=0.7
)

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)

plt.xlabel("Primary tumour vs normal: pathway NES")
plt.ylabel("PDX vs normal: pathway NES")

plt.title(
    "Preservation of tumour-associated biological programmes\n"
    f"Spearman r = {pathway_r:.3f}"
)

plt.tight_layout()

plt.savefig(
    "SRCC_pathway_fidelity.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
effects.to_csv(
    "SRCC_gene_effects_tumour_PDX_vs_normal.csv"
)

pathway_compare.to_csv(
    "SRCC_Hallmark_pathway_fidelity.csv",
    index=False
)

preserved.to_csv(
    "SRCC_preserved_significant_pathways.csv",
    index=False
)

lost.to_csv(
    "SRCC_tumour_significant_PDX_nonsignificant.csv",
    index=False
)

pdx_acquired.to_csv(
    "SRCC_PDX_acquired_pathways.csv",
    index=False
)

direct_pathways.to_csv(
    "SRCC_direct_PDX_vs_tumour_GSEA.csv",
    index=False
)

autophagy_compare.to_csv(
    "SRCC_autophagy_Reactome_comparison.csv",
    index=False
)

print("Final project outputs saved.")

In [ ]:
## 7. Interpretation and limitations

### Main observation

The PDX models retained a highly tumour-like global transcriptional state.

Both PDX samples showed stronger transcriptome-wide similarity to the primary-tumour centroid (Spearman r ≈ 0.987) than to adjacent normal tissue (≈0.89).

Tumour-associated gene-expression changes were also strongly concordant with changes observed in PDX models (Spearman r = 0.964).

At the pathway level, tumour-associated Hallmark enrichment scores were nearly identical between primary tumours and PDX models (Spearman r = 0.998). Thirty-five significant tumour-associated Hallmark programmes were retained in PDX, while no significant PDX-specific Hallmark programmes were detected.

A direct PDX-versus-primary-tumour comparison identified no Hallmark pathway significantly altered at FDR < 0.05. The median absolute transcriptomic difference between PDX and tumour was 0.084 log2(CPM+1), with only eight genes showing an absolute difference greater than 1.

Together, these descriptive results suggest that the available SRCC PDX models retain much of the dominant primary-tumour transcriptional state and pathway-level biology.

### Autophagy-related programmes

Reactome autophagy-related programmes showed similar positive enrichment in both primary tumour and PDX, although broad autophagy gene sets did not reach FDR < 0.05 in this small dataset.

This suggests directional preservation rather than independent statistical confirmation of autophagy activation.

### Important caveat

Both tumour-vs-normal and PDX-vs-normal comparisons share the same normal reference, which can increase apparent concordance. For this reason, model fidelity was also evaluated directly using PDX-versus-primary-tumour expression differences and pathway enrichment.

### Limitations

- Only two samples are available per group.
- This is an exploratory re-analysis of public bulk RNA-seq data.
- PDX models are derived from the tumour, so high transcriptional similarity is expected to some degree.
- Bulk RNA-seq cannot determine whether spatial, stromal, immune or cellular-composition features are preserved.
- No causal or therapeutic conclusion can be established from transcriptomic similarity alone.
- Pathway significance should be interpreted cautiously given the extremely small cohort.